# Field validation — `frontal_structure` (SURF pipeline)

End-to-end validation of every calculated field in the **frontal_structure** surface subset: the notebook RUNs the pipeline, LOADs its own output, and validates each field with dependency-chain maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `frontal_structure` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Every computed step is shown: gradient COMPONENTS (tracer-point, CS/SN-rotated east/north derivatives from `native_gradient.calculate_native_gradient_tracer`) appear as their own chain columns before the squared magnitudes.

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "frontal_structure"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
# get_subset_definition already folds per-pipeline extras (e.g.
# oceQnet for SURF) into model_data_feature_channels.
CHANNELS = (list(defn["model_data_feature_channels"])
            + list(defn["compute_features_channels"]))

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `frontal_structure`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`; no raw model channels — every channel is computed):

`gradb2`, `gradsalt2`, `gradtheta2`, `gradeta2`, `gradrho2`, `turner_angle`, `density`, `buoyancy`

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert set(reader.channel_names) == set(CHANNELS), (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel set matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| density (σ₀) | kg m⁻³ | σ₀ = ρ_JMD95(S, Θ, p=0) − 1000 | Theta, Salt → rho_theta | `calculate_fields.potential_density_anomaly` |
| buoyancy (b) | m s⁻² | b = g·σ₀/ρ₀ | density (← Theta, Salt) | `calculate_fields.buoyancy_of_field` |
| gradtheta2 | °C² m⁻² | \|∇Θ\|² = Θₓ² + Θᵧ² | dTheta_dx, dTheta_dy (← Theta) | `calculate_fields.grad_theta2` → `native_gradient` |
| gradsalt2 | psu² m⁻² | \|∇S\|² | dSalt_dx, dSalt_dy (← Salt) | `calculate_fields.grad_salt2` → `native_gradient` |
| gradeta2 | m² m⁻² | \|∇η\|² | dEta_dx, dEta_dy (← Eta) | `calculate_fields.grad_eta2` → `native_gradient` |
| gradrho2 | kg² m⁻⁸ | \|∇ρθ\|² | drho_dx, drho_dy (← rho_theta ← Theta, Salt) | `calculate_fields.grad_rho2` → `native_gradient` |
| gradb2 | s⁻⁴ | \|∇b\|² | db_dx, db_dy (← buoyancy) | `calculate_fields.grad_b2` → `native_gradient` |
| turner_angle | ° | Tu = arctan[∇ρ·(α∇T + β∇S) / ∇ρ·(α∇T − β∇S)] — projection form, Johnson et al. (2012) / Whalen & Drushka (2025); measured ∇ρ; co-located staggered dot products (`ng.calculate_grad_dot_tracer`); independent of the grad² channels | Theta, Salt → rho_theta | `calculate_fields.turner_angle` |

Non-channel intermediates (computed LIVE below with the same
production code, plotted as chain columns):

| INTERMEDIATE | UNITS | EQUATION | LOCATION |
|---|---|---|---|
| rho_theta | kg m⁻³ | ρ_JMD95(S, Θ, p=0) | `calculate_fields.potential_density` |

**Stencil note (sparkle fix, 2026-08-05)** — the squared magnitudes (all five grad*2) are built SQUARE-BEFORE-INTERP from the native staggered differences (`ng.calculate_grad_squared_tracer`); the displayed gradient components are the geographic (centre-interpolated) vectors — physically the same chain, but the finals are NOT pixelwise combinations of them (see docs/Fields.md).

| d{X}_dx, d{X}_dy for X ∈ Θ, S, η, ρθ, b | per field / m | native tracer gradient + CS/SN rotation | `native_gradient.calculate_native_gradient_tracer` |

Processing operations: land masking (`hFacC == 0`, NaN); native-grid
tracer differentiation with CS/SN rotation (NaN halo rim at face
edges); **no** staggered→tracer interpolation; face→lat-lon
stitching; global row downsampled `[::12, ::12]`.  Conventions:
σ₀ = ρ − 1000; b = g·σ₀/ρ₀ (anomaly-based).

### Raw inputs & live intermediates

The store holds only the 8 output channels.  Raw inputs, rho_theta, and ALL gradient components are computed live from the same OSN snapshot with the same code, batch-stitched, and sliced to the validation domains.

In [ ]:
# Live raw + intermediate fields (same loaders as the pipeline).
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

# Grid + snapshot exactly as generate-global does for SURF.
ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt", "Eta"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}  (store iteration {reader.iteration})")
for _v in ds_merge.data_vars:
    if ds_merge[_v].ndim >= 2:
        print(f"  raw {_v}: {ds_merge[_v].dtype}")

# Lazy live fields: raw, rho_theta, and every gradient component
# (tracer-point, CS/SN-rotated) — the exact code under validation.
rho = calculate_fields.potential_density(ds_merge)
b = calculate_fields.buoyancy_of_field(ds_merge)
gTh = ng.calculate_native_gradient_tracer(
    ds_merge.Theta, ds_merge, grid=xgrid)
gS = ng.calculate_native_gradient_tracer(
    ds_merge.Salt, ds_merge, grid=xgrid)
gE = ng.calculate_native_gradient_tracer(
    ds_merge.Eta, ds_merge, grid=xgrid)
gR = ng.calculate_native_gradient_tracer(rho, ds_merge, grid=xgrid)
gB = ng.calculate_native_gradient_tracer(b, ds_merge, grid=xgrid)

live_map = {
    "Theta": ds_merge["Theta"], "Salt": ds_merge["Salt"],
    "Eta": ds_merge["Eta"], "rho_theta": rho,
    "dTheta_dx": gTh[0], "dTheta_dy": gTh[1],
    "dSalt_dx": gS[0], "dSalt_dy": gS[1],
    "dEta_dx": gE[0], "dEta_dy": gE[1],
    "drho_dx": gR[0], "drho_dy": gR[1],
    "db_dx": gB[0], "db_dy": gB[1],
    # Live finals via the canonical (square-before-interp)
    # functions — consumed by the consistency checks; the store
    # finals CANNOT be reproduced from the centred components
    # (different stencils; see docs/Fields.md).
    "gradtheta2_live": calculate_fields.grad_theta2(ds_merge, xgrid),
    "gradsalt2_live": calculate_fields.grad_salt2(ds_merge, xgrid),
    "gradeta2_live": calculate_fields.grad_eta2(ds_merge, xgrid),
    "gradrho2_live": calculate_fields.grad_rho2(ds_merge, xgrid),
    "gradb2_live": calculate_fields.grad_b2(ds_merge, xgrid),
    "turner_angle_live": calculate_fields.turner_angle(
        ds_merge, xgrid),
}
# Batch-stitch the live fields and slice to the validation domains
# (shared plumbing: dbof.plotting.live_fields; at most `batch`
# full-res arrays alive at once).
from dbof.plotting import regions
from dbof.plotting.live_fields import stitch_and_slice

SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
region_arrays = globals().get("region_arrays", {})
region_arrays.update(stitch_and_slice(
    live_map, ds_raw, ds_merge, XC, YC, SLICE_REGIONS, batch=4))
print(f"live fields ready: {list(live_map)}")

In [ ]:
# Slice the STORE channels to the validation domains (live fields,
# if any, were sliced in the previous cell) via the shared plumbing
# in dbof.plotting.live_fields.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps
from dbof.plotting.live_fields import (
    slice_store, print_region_summary,
)

CMAP_CFG, DIVERGING = load_field_cmaps()

# region_arrays[field][region] = (x, y, arr)
region_arrays = globals().get("region_arrays", {})
SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
region_arrays.update(
    slice_store(reader, CHANNELS, XC, YC, SLICE_REGIONS))
print_region_summary(region_arrays)

## Section 5 — Per-field validation

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → components → final; every computed step is shown), rows =
  validation domains.  One shared colour scale per column; land/halo
  NaNs gray; regional boxes on the global row.
- **Figure 2 — PDFs**: same grid.  Probability density; land +
  halo-rim NaNs removed; bins shared per field across domains;
  Eq. Pacific row |lat|>2° filtered for f-normalised fields.
- **Literature comparisons** live in Section 6 at the end of the
  notebook — one subsection PER REFERENCE (a reference may validate
  several fields at once), only where a reference exists.  Images in
  `../literature_figures/`, named
  `{field(s)}_{Citation}_{description}.png`.

In [ ]:
# Section 5 helpers: one call per figure, shared by all fields.
from pathlib import Path

import cartopy.crs as ccrs

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import (
    pipeline_map_grid, mask_wrap_cells, LAND_COLOR,
)
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

# Flat literature directory; files named
# {field}_{Citation}_{description}.png
LIT_DIR = Path("../literature_figures")

# Full dependency chain per field (columns of Figures 1-2), including
# component-level intermediates (gradient / Jacobian components).
CHAINS = {
    "density": ["Theta", "Salt", "rho_theta", "density"],
    "buoyancy": ["Theta", "Salt", "rho_theta", "density", "buoyancy"],
    "gradtheta2": ["Theta", "dTheta_dx", "dTheta_dy", "gradtheta2"],
    "gradsalt2": ["Salt", "dSalt_dx", "dSalt_dy", "gradsalt2"],
    "gradeta2": ["Eta", "dEta_dx", "dEta_dy", "gradeta2"],
    "gradrho2": ["rho_theta", "drho_dx", "drho_dy", "gradrho2"],
    "gradb2": ["buoyancy", "db_dx", "db_dy", "gradb2"],
    "turner_angle": ["gradtheta2", "gradsalt2", "gradrho2", "turner_angle"],
}

# f-normalised fields: |lat|>2 deg filter on the Eq. Pacific row of
# the PDFs only (maps annotated instead) — plan Clarification 8.
F_NORM = set()

# Fields drawn/binned on log scales (∝-squared fields).
LOG_FIELDS = {"gradb2", "gradeta2", "gradrho2", "gradsalt2", "gradtheta2"}

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; shared bins "
            "across domains; log10-x for \u221d-squared fields")


# Shared figure toolkit (extracted to dbof.plotting.validation_figures
# — one implementation for all six notebooks); methods bound to the
# historical cell-level names used by the Section 5/6 cells below.
from dbof.plotting.validation_figures import ValidationFigures

_figs = ValidationFigures(
    chains=CHAINS, region_arrays=region_arrays,
    cmap_cfg=CMAP_CFG, diverging=DIVERGING,
    log_fields=LOG_FIELDS, f_norm=F_NORM, lit_dir=LIT_DIR,
)
figure1_maps = _figs.maps
figure2_pdfs = _figs.pdfs
figure3_literature = _figs.literature

### 5.1 density (σ₀)

Theta, Salt → rho_theta (JMD95 at p=0) → σ₀ = ρθ − 1000 (`calculate_fields.potential_density_anomaly`).

In [ ]:
figure1_maps("density")

In [ ]:
figure2_pdfs("density")

### 5.2 buoyancy (b)

σ₀ → b = g·σ₀/ρ₀ [m s⁻²] (`calculate_fields.buoyancy_of_field`).

In [ ]:
figure1_maps("buoyancy")

In [ ]:
figure2_pdfs("buoyancy")

### 5.3 gradtheta2 (|∇Θ|²)

Theta → (∂Θ/∂x, ∂Θ/∂y) via native tracer gradient + CS/SN rotation → sum of squares (`calculate_fields.grad_theta2`).  Components should be diverging around 0 with frontal banding.

In [ ]:
figure1_maps("gradtheta2")

In [ ]:
figure2_pdfs("gradtheta2")

### 5.4 gradsalt2 (|∇S|²)

Salt → (∂S/∂x, ∂S/∂y) → |∇S|² (`calculate_fields.grad_salt2`).

In [ ]:
figure1_maps("gradsalt2")

In [ ]:
figure2_pdfs("gradsalt2")

### 5.5 gradeta2 (|∇η|²)

Eta → (∂η/∂x, ∂η/∂y) → |∇η|² (`calculate_fields.grad_eta2`).  Geostrophic KE proxy.

In [ ]:
figure1_maps("gradeta2")

In [ ]:
figure2_pdfs("gradeta2")

### 5.6 gradrho2 (|∇ρ|²)

rho_theta → (∂ρ/∂x, ∂ρ/∂y) → |∇ρθ|² (`calculate_fields.grad_rho2`).

In [ ]:
figure1_maps("gradrho2")

In [ ]:
figure2_pdfs("gradrho2")

### 5.7 gradb2 (|∇b|²)

buoyancy → (∂b/∂x, ∂b/∂y) → |∇b|² (`calculate_fields.grad_b2`).  Structure = gradrho2 scaled by (g/ρ₀)².

In [ ]:
figure1_maps("gradb2")

In [ ]:
figure2_pdfs("gradb2")

### 5.8 turner_angle (Tu)

gradtheta2, gradsalt2, gradrho2 → Tu = arctan[ρ₀(β²|∇S|² − α²|∇Θ|²)/(−|∇ρ|²/ρ₀)] [°] (`calculate_fields.turner_angle`).  Masked where |∇ρ| = 0.

In [ ]:
figure1_maps("turner_angle")

In [ ]:
figure2_pdfs("turner_angle")

## Section 6 — Literature comparisons

One subsection per reference (a reference may validate several fields); only fields with published counterparts appear here.

### 6.1 Whalen & Drushka (2025) — Turner angle, global

Global World Ocean Atlas climatology (image filename says 2024; the
published paper is 2025) vs our single LLC4320 snapshot:
expect agreement in the large-scale Tu regimes, not mesoscale
detail.

In [ ]:
# Whalen & Drushka comparison: Pacific-centred Robinson and THEIR
# colour convention (turbo-like, fixed -90..+90) so the panels read
# side by side.


def _tu_wd_panel(ax):
    """Our global Tu in the W&D display convention.
    Inputs: ax (cartopy axis).  Outputs: draws on ax.
    Generated by LH and Claude
    """
    x, y, arr = region_arrays["turner_angle"]["global"]
    arr = mask_wrap_cells(x, y, arr)
    ax.set_facecolor(LAND_COLOR)
    im = ax.pcolormesh(x, y, arr, cmap="turbo",
                       vmin=-90, vmax=90,
                       transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    plt.colorbar(im, ax=ax, orientation="horizontal",
                 fraction=0.04, pad=0.04,
                 label="Turner angle (deg)")


side_by_side(
    _tu_wd_panel,
    LIT_DIR / ("turner-angle_Whalen&Drushka(2024)_"
               "World-Ocean-Atlas-climatology.png"),
    projection=ccrs.Robinson(central_longitude=180),
    caption=("Compare with Whalen & Drushka (2025), World Ocean "
             "Atlas climatology.  Our panel uses their display "
             "convention (Pacific-centred, turbo, \u00b190\u00b0). "
             "Note: WOA is a coarse multi-decade "
             "climatology vs one LLC4320 snapshot \u2014 expect "
             "agreement in the large-scale Tu pattern (T- vs "
             "S-controlled regimes), not in mesoscale detail.\n"
             "Whalen, C. B., and K. Drushka, 2025: Global "
             "Distribution and Governing Dynamics of "
             "Submesoscale Density Fronts. J. Phys. Oceanogr., "
             "55, 1831\u20131845, "
             "https://doi.org/10.1175/JPO-D-24-0119.1."),
)

### 6.2 Global |∇b| snapshot comparison

|∇b| = √(bₓ² + b_y²) from the live gradient components, global view,
colour scale pinned to the reference (0–1.1e-6 s⁻²).  Rename the
``PNG_NAME`` below to match the saved reference image / citation.

In [ ]:
# Global |grad b| = hypot(db_dx, db_dy) vs literature global map.
import cartopy.crs as ccrs

from dbof.plotting.literature_comparison import side_by_side

PNG_NAME = ("frontal-structure_gradb_global_LLC4320_"
            "Bodner-et-al(2025).png")

_xg, _yg, _dbx = region_arrays["db_dx"]["global"]
_dby = region_arrays["db_dy"]["global"][2]
gradb_glob = mask_wrap_cells(_xg, _yg, np.hypot(_dbx, _dby))


def _gradb_panel(ax):
    """Global |grad b| on the reference colour scale.

    Inputs: ax (cartopy GeoAxes).  Outputs: draws on ax.
    Generated by LH and Claude
    """
    ax.set_facecolor("black")     # reference uses black land/bg
    im = ax.pcolormesh(_xg, _yg, gradb_glob,
                       transform=ccrs.PlateCarree(),
                       cmap="magma", vmin=0.0, vmax=1.1e-6,
                       shading="nearest")
    ax.coastlines(linewidth=0.3, color="gray")
    plt.colorbar(im, ax=ax, orientation="horizontal",
                 fraction=0.04, pad=0.04, extend="max",
                 label="|∇b| (s⁻²) "
                       "[ref scale 0–1.1e-6]")


side_by_side(
    _gradb_panel, LIT_DIR / PNG_NAME,
    projection=ccrs.Robinson(),
    caption=("Global surface |∇b| = √(bₓ² + "
             "bᵧ²), colour scale pinned to the "
             "reference.  Expect WBC extensions, ACC, and "
             "equatorial fronts bright; gyre interiors dark.\n"
             "Bodner, A., Balwada, D., & Zanna, L. (2025). A "
             "data-driven approach for parameterizing ocean "
             "submesoscale buoyancy fluxes. J. Adv. Model. "
             "Earth Syst., 17, e2025MS004991. "
             "https://doi.org/10.1029/2025MS004991."),
)
plt.show()

### 6.3 Pressure-gradient force g·|∇η| comparison

g·√(gradeta2) from the STORE channel (validates the product
directly), colour scale pinned to the reference (0–8e-5 m s⁻²).
NOTE: the reference is a time-RMS ⟨g∇η⟩_rms (internal-tide beams
smoothed/emphasised); ours is ONE snapshot — expect the same
large-scale pattern but streakier instantaneous detail.  Rename
``PNG_NAME`` to match the saved reference image / citation.

In [ ]:
# Global pressure-gradient force g*|grad eta| vs literature RMS map.
from dbof.preprocessing.physical_constants import G

PNG_NAME = ("frontal-structure_gradEta_globalLLC4320_"
            "Yu-et-al(2021).png")

_xg2, _yg2, _ge2 = region_arrays["gradeta2"]["global"]
pgf_glob = mask_wrap_cells(_xg2, _yg2, G * np.sqrt(_ge2))


def _pgf_panel(ax):
    """Global g|grad eta| on the reference colour scale.

    Inputs: ax (cartopy GeoAxes).  Outputs: draws on ax.
    Generated by LH and Claude
    """
    ax.set_facecolor(LAND_COLOR)
    im = ax.pcolormesh(_xg2, _yg2, pgf_glob,
                       transform=ccrs.PlateCarree(),
                       cmap="OrRd", vmin=0.0, vmax=8e-5,
                       shading="nearest")
    ax.coastlines(linewidth=0.3, color="k")
    plt.colorbar(im, ax=ax, orientation="horizontal",
                 fraction=0.04, pad=0.04, extend="max",
                 label="g·|∇η| (m s⁻²) "
                       "[ref scale 0–8e-5]")


side_by_side(
    _pgf_panel, LIT_DIR / PNG_NAME,
    projection=ccrs.Robinson(),
    caption=("Instantaneous g·|∇η| from the "
             "gradeta2 store channel vs the reference time-RMS "
             "⟨g∇η⟩: same large-scale "
             "pattern (WBCs, ACC, internal-tide generation "
             "sites); a single snapshot is streakier than the "
             "RMS.\n"
             "Yu, X., Ponte, A. L., Lahaye, N., Caspar-Cohen, Z., "
             "& Menemenlis, D. (2021). Geostrophy assessment and "
             "momentum balance of the global oceans in a tide- and "
             "eddy-resolving model. J. Geophys. Res.: Oceans, 126, "
             "e2021JC017422. "
             "https://doi.org/10.1029/2021JC017422."),
)
plt.show()

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':24s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    key = hash(sub[finite][::997].tobytes()) if finite.any() else ch
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:24s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

In [ ]:
# Store-vs-live consistency: each final channel (store) must equal
# the function of its live-computed dependencies (Gulf Stream domain;
# land + halo-rim NaNs excluded).  This turns the two-path design
# (finals via RUN->store->LOAD, dependencies via live compute) into
# an explicit pass/fail test of the pipeline plumbing.
from dbof.preprocessing.physical_constants import (
    G, RHO0_REFERENCE, ALPHA, BETA,
)


def _gs(field):
    """Gulf Stream slice of one field.

    Inputs: field (str).  Outputs: 2D np.ndarray.
    Generated by LH and Claude
    """
    return region_arrays[field]["gulf_stream"][2]


CHECKS = {
    # grad*2 are square-BEFORE-interp (sparkle fix, 2026-08-05):
    # NOT reproducible from the centred components — compare the
    # store against the live canonical function instead.
    "gradtheta2 (store) = grad_theta2 (live sq-first)":
        (_gs("gradtheta2"), _gs("gradtheta2_live")),
    "gradsalt2 (store) = grad_salt2 (live sq-first)":
        (_gs("gradsalt2"), _gs("gradsalt2_live")),
    "gradeta2 (store) = grad_eta2 (live sq-first)":
        (_gs("gradeta2"), _gs("gradeta2_live")),
    "gradrho2 (store) = grad_rho2 (live sq-first)":
        (_gs("gradrho2"), _gs("gradrho2_live")),
    "gradb2 (store) = grad_b2 (live sq-first)":
        (_gs("gradb2"), _gs("gradb2_live")),
    "density = rho_theta - RHO0":
        (_gs("density"), _gs("rho_theta") - RHO0_REFERENCE),
    "buoyancy = G*density/RHO0":
        (_gs("buoyancy"), G * _gs("density") / RHO0_REFERENCE),
    # turner_angle is the projection form (Johnson et al. 2012,
    # plan 2026-08-06): built from measured-grad(rho) dot products,
    # NOT from the grad² channels — compare store vs live canonical.
    "turner_angle (store) = turner_angle (live, projection)":
        (_gs("turner_angle"), _gs("turner_angle_live")),
}

# Shared pass/fail loop (dbof.plotting.validation_figures);
# rel_tol=1e-4 — float32 store vs float64 live recompute.
from dbof.plotting.validation_figures import run_consistency_checks

run_consistency_checks(CHECKS)

**Cross-references** — σ₀ and b are validated here; sibling notebooks (`kinematic`, `frontogenesis`, DEPTH `stratification`, ...) reference these sections for the shared density/buoyancy machinery rather than re-validating it.